# Analyser ses données trail (2/3) — Comprendre le terrain

Notebook associé au billet : https://gregs1t.github.io/trail/2026-04-01-trail-analyse-2-terrain/

**Pré-requis** : avoir exécuté `01_lire_donnees.ipynb` — le DataFrame `df` doit être en mémoire,
avec les colonnes `dist_m`, `alt_m`, `speed_kmh`, `pace_s_per_km`.

**Ce qu'on fait ici :**
- Pente locale robuste sur fenêtre glissante
- Segmentation montées / descentes / plat avec hystérésis
- Classification marche vs course (vitesse + cadence)
- Grade Adjusted Pace (GAP) via le polynôme de Minetti (Minetti, 2002)
- VAM et résumé par section entre ravitaillements


Les notions qui sont abordées ici sont décrites dans le billet du blog.
J'y ai aussi mis les références scienfiques à la fin.



### 
### © Gregory Sainton
### 
- Notebook        : Analyse d'une séance de trail/course 
- Auteur          : Grégory Sainton <twinity4trail@proton.me>
- Dépôt           : https://github.com/gregs1t/trail-lab
- Date de création: 2025-12
- Dernière modif. : 2026-04
- Version         : 1.0
---
- Licence         : CC BY-NC-SA 4.0
-                   https://creativecommons.org/licenses/by-nc-sa/4.0/

>    Vous êtes libre de partager et d'adapter ce travail, à condition de :
>     · citer l'auteur (BY)
>     · ne pas en faire un usage commercial (NC)
>     · redistribuer sous la même licence (SA)

## ⚙️ Paramètres supplémentaires

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fitparse import FitFile

pd.set_option("display.max_columns", 30)

# --- À adapter à ta course ---
FIT_PATH = "mon_fichier_a_moi.fit"   # chemin vers ton fichier .fit




# Ces paramètres doivent aussi être définis (copie depuis le notebook 1 si besoin)
FC_MAX = 185
FC_MIN = 47
RAVITO_KM  = [19.2, 34.0, 45.0, 58.8, 65.4]
RAVITO_NOM = ["St Christo", "Ste Catherine", "St Genou", "Soucieu", "Chaponost"]

# Nouveaux paramètres pour ce notebook
WINDOW_M     = 100.0   # fenêtre pente en mètres (50–200)
UP_THR       = +3.0    # seuil montée (%)
DOWN_THR     = -3.0    # seuil descente (%)
MIN_SEG_M    = 200.0   # longueur minimale de segment (m)
WALK_THR_KMH = 6.0     # seuil vitesse marche (km/h)
WALK_THR_CAD = 140.0   # seuil cadence marche (pas/min)

In [ ]:
def load_fit(fit_path):
    """Load FIT file records into a pandas DataFrame."""
    fitfile = FitFile(fit_path)
    records = []
    for record in fitfile.get_messages("record"):
        row = {}
        for field in record:
            row[field.name] = field.value
        records.append(row)
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["time_h"] = (
        (df["timestamp"] - df["timestamp"].iloc[0])
        .dt.total_seconds() / 3600.0
    )
    return df


df = load_fit(FIT_PATH)
print(f"{len(df)} points enregistrés")
print(f"Colonnes disponibles : {df.columns.tolist()}")

In [ ]:
def clean_df(df):
    """Select best columns, enforce monotone distance, compute pace."""
    alt_col = "enhanced_altitude" if "enhanced_altitude" in df.columns else "altitude"
    spd_col = "enhanced_speed" if "enhanced_speed" in df.columns else "speed"

    df = df.dropna(subset=["distance", alt_col]).reset_index(drop=True)

    df["dist_m"] = np.maximum.accumulate(df["distance"].to_numpy(dtype=float))
    df["alt_m"] = df[alt_col].to_numpy(dtype=float)

    if spd_col in df.columns:
        v = df[spd_col].to_numpy(dtype=float)
        df["speed_mps"] = v
        df["speed_kmh"] = v * 3.6
        df["pace_s_per_km"] = np.where(v > 0.5, 1000.0 / v, np.nan)
    else:
        df["speed_mps"] = np.nan
        df["speed_kmh"] = np.nan
        df["pace_s_per_km"] = np.nan

    return df


df = clean_df(df)
df[["dist_m", "alt_m", "speed_kmh", "pace_s_per_km"]].describe().round(1)

## 1. Pente locale robuste

La pente point-à-point est inexploitable (bruit GPS). On calcule sur une fenêtre arrière de `WINDOW_M` mètres.

In [ ]:
def compute_slope(df, window_m):
    """Compute local slope (%) over a backward distance window."""
    d = df["dist_m"].to_numpy()
    z = df["alt_m"].to_numpy()
    j = np.searchsorted(d, d - window_m, side="left")
    j = np.clip(j, 0, len(d) - 1)
    dd = d - d[j]
    dz = z - z[j]
    return np.where(dd > 0, (dz / dd) * 100.0, np.nan)


df["slope_pct"] = compute_slope(df, WINDOW_M)
print(df["slope_pct"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(1))

In [ ]:
# Vérification visuelle : profil + pente alignés
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(df["dist_m"] / 1000.0, df["alt_m"], color="brown", linewidth=1.5)
axes[0].fill_between(df["dist_m"] / 1000.0, df["alt_m"],
                     df["alt_m"].min(), color="brown", alpha=0.2)
axes[0].set_ylabel("Altitude (m)")
axes[0].set_title("Profil altimétrique")
axes[0].grid(True)

axes[1].scatter(df["dist_m"] / 1000.0, df["slope_pct"],
                s=2, alpha=0.4, color="steelblue")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].axhline(UP_THR, color="red", linewidth=0.8, linestyle="--", alpha=0.5,
                label=f"Seuil montée ({UP_THR} %)")
axes[1].axhline(DOWN_THR, color="blue", linewidth=0.8, linestyle="--", alpha=0.5,
                label=f"Seuil descente ({DOWN_THR} %)")
axes[1].set_xlabel("Distance (km)")
axes[1].set_ylabel("Pente (%)")
axes[1].set_title(f"Pente locale (fenêtre {WINDOW_M:.0f} m)")
axes[1].legend(fontsize=8)
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 2. Segmentation montées / descentes

Hystérésis pour éviter les oscillations dans les zones de transition + longueur minimale pour éliminer les faux segments.

In [ ]:
def segment_updown(df, up_thr, down_thr, min_seg_m):
    """Segment track into uphill (+1), downhill (-1), flat (0) with hysteresis."""
    s = df["slope_pct"].to_numpy()
    state = np.zeros(len(df), dtype=int)
    state[s >= up_thr] = 1
    state[s <= down_thr] = -1

    for i in range(1, len(state)):
        if state[i] == 0:
            state[i] = state[i - 1]

    df = df.copy()
    df["ud_state"] = state
    df["seg_id"] = (df["ud_state"] != df["ud_state"].shift(1)).cumsum()

    seg_len = df.groupby("seg_id")["dist_m"].agg(
        lambda x: float(x.iloc[-1] - x.iloc[0])
    )
    valid = seg_len[seg_len >= min_seg_m].index
    df["ud_clean"] = np.where(df["seg_id"].isin(valid), df["ud_state"], 0)
    return df


df = segment_updown(df, UP_THR, DOWN_THR, MIN_SEG_M)
print(df["ud_clean"].value_counts().rename({1: "montée", -1: "descente", 0: "plat"}))

In [ ]:
# Profil coloré : rouge = montée, bleu = descente, vert = plat
colors = {1: "tab:red", -1: "tab:blue", 0: "tab:green"}
labels_ud = {1: "Montée", -1: "Descente", 0: "Plat"}

fig, ax = plt.subplots(figsize=(13, 5))
for state, color in colors.items():
    mask = df["ud_clean"] == state
    ax.scatter(df.loc[mask, "dist_m"] / 1000.0, df.loc[mask, "alt_m"],
               s=3, alpha=0.5, color=color, label=labels_ud[state])
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Altitude (m)")
ax.set_title("Segmentation montée / descente / plat")
ax.legend(markerscale=4)
ax.grid(True)
plt.tight_layout()
plt.show()

## 3. Classification marche vs course

Double critère vitesse + cadence : plus robuste que la vitesse seule en trail.

In [ ]:
def classify_walk_run(df, walk_thr_kmh, walk_thr_cad):
    """Classify each point as walking (1) or running (0)."""
    df = df.copy()
    if "cadence" not in df.columns:
        print("Pas de cadence — classification impossible.")
        df["is_walk"] = np.nan
        return df
    df["is_walk"] = (
        (df["speed_kmh"] < walk_thr_kmh) & (df["cadence"] < walk_thr_cad)
    ).astype(int)
    return df


df = classify_walk_run(df, WALK_THR_KMH, WALK_THR_CAD)

if df["is_walk"].notna().any():
    print(f"Temps en marche : {df['is_walk'].mean() * 100:.1f} %")

In [ ]:
# Proportion de marche par classe de pente — 1ère vs 2ème moitié
if "is_walk" in df.columns and df["is_walk"].notna().any():
    df["slope_bin"] = pd.cut(
        df["slope_pct"],
        bins=[-30, -10, -3, 3, 10, 20, 40],
        labels=["< -10 %", "-10/-3 %", "plat", "+3/+10 %", "+10/+20 %", "> +20 %"]
    )
    mid_km = df["dist_m"].max() / 2000.0
    df["half"] = np.where(
        df["dist_m"] / 1000.0 < mid_km, "1ère moitié", "2ème moitié"
    )

    walk_by_slope = (
        df.dropna(subset=["slope_bin", "cadence"])
        .groupby(["slope_bin", "half"], observed=True)["is_walk"]
        .mean() * 100
    ).unstack()

    fig, ax = plt.subplots(figsize=(9, 4))
    walk_by_slope.plot(kind="bar", ax=ax, edgecolor="white")
    ax.set_ylabel("% de marche")
    ax.set_title("Marche par classe de pente — 1ère vs 2ème moitié de course")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_walk_by_slope_sections(df, slope_bins, slope_labels,
                                ravito_km=None, n_segments=None,
                                walk_thr_kmh=6.0, walk_thr_cad=140.0):
    """Plot walk probability by slope class, split by race sections.
 
    Sections are defined either by aid station positions (ravito_km) or by
    splitting the track into n equal-distance segments (n_segments).
    Exactly one of the two must be provided.
 
    Parameters
    ----------
    df : DataFrame with dist_m, slope_pct, cadence, speed_kmh, is_walk columns.
    slope_bins : list of bin edges for pd.cut (e.g. [-30, -10, -3, 3, 10, 20, 40]).
    slope_labels : list of str labels (len = len(slope_bins) - 1).
    ravito_km : list of float, aid station positions in km. Optional.
    n_segments : int, number of equal-distance segments. Optional.
    walk_thr_kmh : float, speed threshold for walking classification.
    walk_thr_cad : float, cadence threshold for walking classification.
    """
    if ravito_km is None and n_segments is None:
        raise ValueError("Provide either ravito_km or n_segments.")
    if ravito_km is not None and n_segments is not None:
        raise ValueError("Provide only one of ravito_km or n_segments, not both.")
 
    df = df.copy()
 
    # Classify walk/run if not already done
    if "is_walk" not in df.columns:
        df["is_walk"] = (
            (df["speed_kmh"] < walk_thr_kmh) & (df["cadence"] < walk_thr_cad)
        ).astype(int)
 
    # Build section boundaries and labels
    dist_min_km = df["dist_m"].min() / 1000.0
    dist_max_km = df["dist_m"].max() / 1000.0
 
    if ravito_km is not None:
        bounds = [dist_min_km] + list(ravito_km) + [dist_max_km]
    else:
        bounds = list(
            np.linspace(dist_min_km, dist_max_km, n_segments + 1)
        )
 
    section_labels = [
        f"{bounds[i]:.1f}–{bounds[i+1]:.1f} km"
        for i in range(len(bounds) - 1)
    ]
 
    # Assign section to each point
    df["section"] = pd.cut(
        df["dist_m"] / 1000.0,
        bins=bounds,
        labels=section_labels,
        include_lowest=True
    )
 
    # Slope bins
    df["slope_bin"] = pd.cut(
        df["slope_pct"],
        bins=slope_bins,
        labels=slope_labels
    )
 
    # Compute walk probability per (section, slope_bin)
    tmp = df.dropna(subset=["section", "slope_bin", "cadence"])
    walk_matrix = (
        tmp.groupby(["slope_bin", "section"], observed=True)["is_walk"]
        .mean() * 100
    ).unstack("section")
 
    # Plot
    n_sections = len(section_labels)
    fig, ax = plt.subplots(figsize=(max(8, 2 * n_sections), 5))
    walk_matrix.plot(kind="bar", ax=ax, edgecolor="white", width=0.8)
 
    ax.set_ylabel("% de marche")
    ax.set_xlabel("Classe de pente")
    title = (
        "Probabilité de marcher par classe de pente — sections ravitos"
        if ravito_km is not None
        else f"Probabilité de marcher par classe de pente — {n_segments} segments"
    )
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Section", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()
 
SLOPE_BINS   = [-30, -10, -3, 3, 10, 20, 40]
SLOPE_LABELS = ["< -10%", "-10/-3%", "plat", "+3/+10%", "+10/+20%", "> +20%"]
 
# Mode 1 : découpage par ravitos
plot_walk_by_slope_sections(
    df,
    slope_bins=SLOPE_BINS,
    slope_labels=SLOPE_LABELS,
    ravito_km=RAVITO_KM,   # liste définie dans les paramètres du notebook
)
 
# Mode 2 : découpage en n segments égaux (ex. 3 tiers de course)
plot_walk_by_slope_sections(
    df,
    slope_bins=SLOPE_BINS,
    slope_labels=SLOPE_LABELS,
    n_segments=3,
)

## 4. Grade Adjusted Pace (GAP)

Allure corrigée du dénivelé via le polynôme de Minetti et al. (2002).
Permet de comparer l'effort entre sections de profils très différents.

In [ ]:
def minetti_cost_ratio(slope_pct):
    """Cost ratio relative to flat ground from Minetti et al. (2002)."""
    i = slope_pct / 100.0
    cr = (155.4 * i**5 - 30.4 * i**4 - 43.3 * i**3
          + 46.3 * i**2 + 19.5 * i + 3.6)
    return np.clip(cr / 3.6, 0.1, None)


def compute_gap(df):
    """Compute grade adjusted pace (s/km) using Minetti polynomial."""
    df = df.copy()
    ratio = minetti_cost_ratio(df["slope_pct"].to_numpy())
    df["gap_s_per_km"] = df["pace_s_per_km"] / ratio
    return df


df = compute_gap(df)
print(f"Allure brute méd. : {df['pace_s_per_km'].median():.0f} s/km")
print(f"GAP méd.          : {df['gap_s_per_km'].median():.0f} s/km")

In [ ]:
# Comparaison allure brute vs GAP sur le profil
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for ax, col, title in [
    (axes[0], "pace_s_per_km", "Allure brute (s/km)"),
    (axes[1], "gap_s_per_km",  "GAP — allure corrigée (s/km)"),
]:
    mask = df[col].notna()
    sc = ax.scatter(df.loc[mask, "dist_m"] / 1000.0,
                    df.loc[mask, "alt_m"],
                    c=df.loc[mask, col],
                    cmap="RdYlGn_r", s=4, alpha=0.8, vmin=180, vmax=900)
    ax.fill_between(df["dist_m"] / 1000.0, df["alt_m"], df["alt_m"].min(),
                    color="lightgrey", alpha=0.3, zorder=0)
    plt.colorbar(sc, ax=ax, label="s/km")
    ax.set_ylabel("Altitude (m)")
    ax.set_title(title)
    ax.grid(True)

axes[-1].set_xlabel("Distance (km)")
plt.tight_layout()
plt.show()

## 5. VAM et résumé par section

VAM = Vitesse Ascendante Moyenne (m/h) — indicateur universel de performance en montée,
comparable entre des sections de profils très différents.

In [ ]:
def section_summary(df, ravito_km, ravito_nom):
    """Compute per-section stats: distance, D+, duration, VAM, GAP, HR, walk ratio."""
    bounds = np.concatenate((
        [float(df["dist_m"].min() / 1000.0)],
        np.array(ravito_km, dtype=float),
        [float(df["dist_m"].max() / 1000.0)]
    ))
    labels = []
    for i in range(len(bounds) - 1):
        if i == 0:
            labels.append(f"Départ → {ravito_nom[0]}")
        elif i == len(bounds) - 2:
            labels.append(f"{ravito_nom[-1]} → Arrivée")
        else:
            labels.append(f"{ravito_nom[i-1]} → {ravito_nom[i]}")

    df = df.copy()
    df["section_id"] = np.searchsorted(
        bounds[1:], df["dist_m"].to_numpy() / 1000.0, side="right"
    )

    rows = []
    for i, lbl in enumerate(labels):
        sec = df[df["section_id"] == i]
        if len(sec) < 10:
            continue
        alt = sec["alt_m"].to_numpy()
        dz = np.diff(alt)
        dplus_sec = float(np.clip(dz, 0, None).sum())
        dur_h = float(sec["time_h"].iloc[-1] - sec["time_h"].iloc[0])
        row = {
            "Section": lbl,
            "Dist (km)": round((sec["dist_m"].iloc[-1] - sec["dist_m"].iloc[0]) / 1000, 1),
            "D+ (m)": round(dplus_sec, 0),
            "Durée (h)": round(dur_h, 2),
            "VAM (m/h)": round(dplus_sec / dur_h, 0) if dur_h > 0 else np.nan,
        }
        if "gap_s_per_km" in sec.columns:
            row["GAP méd. (s/km)"] = round(sec["gap_s_per_km"].median(), 0)
        if "heart_rate" in sec.columns:
            row["FC méd. (bpm)"] = round(sec["heart_rate"].median(), 0)
        if "is_walk" in sec.columns and sec["is_walk"].notna().any():
            row["Marche (%)"] = round(sec["is_walk"].mean() * 100, 1)
        rows.append(row)

    return pd.DataFrame(rows)


if len(RAVITO_KM) > 0:
    stats = section_summary(df, RAVITO_KM, RAVITO_NOM)
    print(stats.to_string(index=False))
else:
    print("Aucun ravitaillement défini — modifier RAVITO_KM dans les paramètres.")

---
**Suite** : notebook `03_cartographie.ipynb` — représenter ses données sur une carte.